# 1. Business Understanding

## 1.1 Бизнесийн асуудал

Энэхүү даалгаврын зорилго нь 31 хоногийн хугацаа хэтэрсэн
зээлдэгч 91 хоногийн хугацаа хэтрүүлэх эсэхийг таамаглах
Machine Learning model боловсруулах.

## 1.2 Target

Target variable нь `is_dpd_91` байна.

- `1` → зээлдэгч DPD91 болсон
- `0` → DPD91 болохоос өмнө төлөлт хийсэн

## 1.3 Prediction Point

`date_dpd_31`-ийг Prediction Point гэж үзнэ.

Өөрөөр хэлбэл model нь тухайн зээлдэгч DPD31 болсон
мөчид available байсан information дээр үндэслэн
DPD91 болох магадлалыг тооцно.

# 2. Data Understanding

## 2.1 Data Loading

In [11]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

FILE_PATH = "../data/raw/Toki Data Scientist task.xlsx"

user_list = pd.read_excel(
    FILE_PATH,
    sheet_name="user_list",
    engine="openpyxl"
)

print("Shape:", user_list.shape)

Shape: (61200, 43)


## 2.2 Dataset-ийн бүтэц

In [12]:
print("Shape:", user_list.shape)

Shape: (61200, 43)


In [13]:
column_info = pd.DataFrame({
    "column": user_list.columns,
    "dtype": user_list.dtypes.astype(str)
})

column_info

,column,dtype
created_month,created_month,int64
date_dpd_31,date_dpd_31,datetime64[ns]
credit_id,credit_id,int64
limit_dpd31,limit_dpd31,float64
balance_dpd31,balance_dpd31,float64
limit_ondue,limit_ondue,int64
balance_ondue,balance_ondue,float64
invoiced_unpaid_amt,invoiced_unpaid_amt,float64
total_payment_amt,total_payment_amt,float64
ostype,ostype,float64


### 2.2.1 Feature-ийн үндсэн бүлгүүд

`user_list` dataset-ийн feature-үүдийг үндсэндээ credit/balance,
user characteristics, app and transaction behavior болон user activity
гэсэн бүлгүүдэд ангилж болно.

`date_dpd_31` нь DPD31 болсон яг огноог илэрхийлэх бөгөөд dataset-ийн
бусад feature-үүдийг энэ өдрийг Day 0 буюу cutoff point гэж үзэн
тодорхойлсон байна.

Иймээс эдгээр feature-үүдийг prediction point дээр available байсан
information гэж үзэж, дараагийн шатанд Data Leakage-ийн эрсдэлийг
нэмэлтээр шалгана.

In [14]:
missing_info = pd.DataFrame({
    "missing_count": user_list.isna().sum(),
    "missing_percent": user_list.isna().mean() * 100
})

missing_info = (
    missing_info
    .sort_values("missing_percent", ascending=False)
)

missing_info

,missing_count,missing_percent
avg_trx_amount_l7d,52655,86.037582
avg_trx_amount_l30d,41502,67.813725
avg_trx_amount_l60d,21144,34.549020
avg_trx_amount_l90d,20073,32.799020
app_recency_days,11522,18.826797
sessions_l7d,11522,18.826797
sessions_l30d,11522,18.826797
sessions_l60d,11522,18.826797
sessions_l90d,11522,18.826797
read_noti_l7d,11522,18.826797


In [15]:
missing_by_month = (
    user_list
    .assign(
        month=user_list["date_dpd_31"].dt.to_period("M").astype(str)
    )
    .groupby("month")
    .agg(
        total_records=("credit_id", "size"),
        missing_target=("is_dpd_91", lambda x: x.isna().sum()),
        missing_limit_dpd31=("limit_dpd31", lambda x: x.isna().sum()),
        missing_balance_dpd31=("balance_dpd31", lambda x: x.isna().sum()),
        missing_age=("age", lambda x: x.isna().sum()),
        missing_sessions_l30d=("sessions_l30d", lambda x: x.isna().sum()),
        missing_avg_trx_l7d=("avg_trx_amount_l7d", lambda x: x.isna().sum())
    )
    .reset_index()
)

missing_by_month

,month,total_records,missing_target,missing_limit_dpd31,missing_balance_dpd31,missing_age,missing_sessions_l30d,missing_avg_trx_l7d
0,2025-12,8796,139,0,0,139,774,7554
1,2026-01,7846,132,0,0,130,725,6788
2,2026-02,8733,103,0,0,102,558,7212
3,2026-03,9775,104,9775,9775,101,589,8263
4,2026-04,9263,93,0,0,93,543,7590
5,2026-05,8943,48,0,0,48,489,7404
6,2026-06,7844,7844,0,0,7844,7844,7844


## 2.3 Missing Values

Dataset-ийн feature-үүдийн missing rate харилцан адилгүй байна.
Зарим behavioral feature өндөр missing rate-тэй байгаа бол зарим
үндсэн feature-д missing утга байхгүй байна.

Missing утгуудын тархалт нь тодорхой time-based pattern-тэй байна.
Тухайлбал, 2026-06 сарын бүх observation-д `is_dpd_91` болон зарим
behavioral feature missing байгаа бол 2026-03 сарын бүх observation-д
`limit_dpd31` болон `balance_dpd31` missing байна.

Иймээс missing value-үүдийг шууд impute хийхээс өмнө үүссэн шалтгаан
болон бизнесийн утгыг шалгана.

In [16]:
trx_missing_check = user_list.groupby(
    user_list["avg_trx_amount_l7d"].isna()
).agg(
    avg_trx_amount_l7d_missing=("credit_id", "size"),
    trx_l7d_mean=("trx_l7d", "mean"),
    trx_l7d_median=("trx_l7d", "median"),
    trx_l7d_zero_rate=("trx_l7d", lambda x: (x == 0).mean() * 100)
)

trx_missing_check

,avg_trx_amount_l7d_missing,trx_l7d_mean,trx_l7d_median,trx_l7d_zero_rate
avg_trx_amount_l7d,,,,
False,8545,4.260503,3.0,0.000000
True,52655,0.000000,0.0,79.720824


In [17]:
sessions_missing_check = user_list.groupby(
    user_list["sessions_l30d"].isna()
).agg(
    records=("credit_id", "size"),
    trx_l30d_mean=("trx_l30d", "mean"),
    trx_l30d_zero_rate=("trx_l30d", lambda x: (x == 0).mean() * 100)
)

sessions_missing_check

,records,trx_l30d_mean,trx_l30d_zero_rate
sessions_l30d,,,
False,49678,3.368294,58.895286
True,11522,0.581906,13.591390


In [18]:
avg_trx_missing_nonzero = user_list[
    user_list["avg_trx_amount_l7d"].isna() &
    (user_list["trx_l7d"] > 0)
]

print("Missing avg_trx_amount_l7d but trx_l7d > 0:",
      len(avg_trx_missing_nonzero))

print("\nBy month:")
print(
    avg_trx_missing_nonzero["date_dpd_31"]
    .dt.to_period("M")
    .value_counts()
    .sort_index()
)

Missing avg_trx_amount_l7d but trx_l7d > 0: 0

By month:
Series([], Freq: M, Name: count, dtype: int64)


In [19]:
avg_trx_missing_nonzero[
    [
        "credit_id",
        "date_dpd_31",
        "trx_l7d",
        "avg_trx_amount_l7d"
    ]
].head(20)

,credit_id,date_dpd_31,trx_l7d,avg_trx_amount_l7d


### Missing pattern-ийн эхний ажиглалт

`avg_trx_amount_l7d` missing байгаа бүх observation-д `trx_l7d = 0`
байна. Иймээс уг missing value нь сүүлийн 7 хоногт transaction
хийгээгүйтэй холбоотой structural missing гэж үзэх үндэслэлтэй.

Иймээс preprocessing хийх үед эдгээр missing утгыг 0 болгон
боловсруулах боломжтой гэж үзэв.

Харин `sessions_l30d`-ийн missing pattern нь transaction inactivity-тэй
шууд нийцэхгүй байгаа тул уг feature-ийн missing утгыг тусад нь
авч үзнэ.

In [20]:
avg_trx_cols = [
    "avg_trx_amount_l7d",
    "avg_trx_amount_l30d",
    "avg_trx_amount_l60d",
    "avg_trx_amount_l90d"
]

for col in avg_trx_cols:
    window = col.replace("avg_trx_amount_", "")
    trx_col = f"trx_{window}"
    
    missing_mask = user_list[col].isna()
    
    print(f"\n{col}")
    print(f"Missing: {missing_mask.sum():,}")
    print(
        f"Missing + corresponding trx > 0: "
        f"{((missing_mask) & (user_list[trx_col] > 0)).sum():,}"
    )
    print(
        f"Missing + corresponding trx = 0: "
        f"{((missing_mask) & (user_list[trx_col] == 0)).sum():,}"
    )


avg_trx_amount_l7d
Missing: 52,655
Missing + corresponding trx > 0: 0
Missing + corresponding trx = 0: 41,977

avg_trx_amount_l30d
Missing: 41,502
Missing + corresponding trx > 0: 0
Missing + corresponding trx = 0: 30,824

avg_trx_amount_l60d
Missing: 21,144
Missing + corresponding trx > 0: 0
Missing + corresponding trx = 0: 10,466

avg_trx_amount_l90d
Missing: 20,073
Missing + corresponding trx > 0: 0
Missing + corresponding trx = 0: 9,395


In [21]:
trx_cols = [
    "trx_l7d",
    "trx_l30d",
    "trx_l60d",
    "trx_l90d"
]

for col in trx_cols:
    missing_mask = user_list[col].isna()

    print(f"\n{col}")
    print(f"Missing: {missing_mask.sum():,}")
    print(
        f"Missing + avg_trx_amount is also missing: "
        f"{(
            missing_mask &
            user_list[f'avg_trx_amount_{col.replace("trx_", "")}'].isna()
        ).sum():,}"
    )


trx_l7d
Missing: 10,678
Missing + avg_trx_amount is also missing: 10,678

trx_l30d
Missing: 10,678
Missing + avg_trx_amount is also missing: 10,678

trx_l60d
Missing: 10,678
Missing + avg_trx_amount is also missing: 10,678

trx_l90d
Missing: 10,678
Missing + avg_trx_amount is also missing: 10,678


In [22]:
trx_missing_by_month = (
    user_list[user_list["trx_l7d"].isna()]
    .assign(
        month=lambda df: df["date_dpd_31"].dt.to_period("M").astype(str)
    )
    .groupby("month")
    .size()
    .reset_index(name="missing_trx_l7d")
)

trx_missing_by_month

,month,missing_trx_l7d
0,2025-12,1032
1,2026-01,972
2,2026-02,330
3,2026-03,233
4,2026-04,163
5,2026-05,104
6,2026-06,7844


In [23]:
trx_missing_profile = (
    user_list[user_list["trx_l7d"].isna()]
    [
        [
            "trx_l7d",
            "trx_l30d",
            "trx_l60d",
            "trx_l90d",
            "trx_recency_l90d",
            "sessions_l7d",
            "sessions_l30d",
            "sessions_l60d",
            "sessions_l90d"
        ]
    ]
    .describe()
    .T
)

trx_missing_profile

,count,mean,std,min,25%,50%,75%,max
trx_l7d,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
trx_l30d,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
trx_l60d,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
trx_l90d,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
trx_recency_l90d,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
sessions_l7d,1024.0,0.341797,0.914941,0.0,0.0,0.0,0.0,9.0
sessions_l30d,1024.0,1.845703,3.320251,0.0,0.0,1.0,2.0,38.0
sessions_l60d,1024.0,2.977539,4.215190,0.0,0.0,2.0,4.0,42.0
sessions_l90d,1024.0,2.536133,3.595460,0.0,0.0,1.0,4.0,41.0


In [24]:
trx_missing_profile

,count,mean,std,min,25%,50%,75%,max
trx_l7d,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
trx_l30d,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
trx_l60d,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
trx_l90d,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
trx_recency_l90d,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
sessions_l7d,1024.0,0.341797,0.914941,0.0,0.0,0.0,0.0,9.0
sessions_l30d,1024.0,1.845703,3.320251,0.0,0.0,1.0,2.0,38.0
sessions_l60d,1024.0,2.977539,4.215190,0.0,0.0,2.0,4.0,42.0
sessions_l90d,1024.0,2.536133,3.595460,0.0,0.0,1.0,4.0,41.0


In [26]:
behavior_cols = [
    "trx_recency_l90d",
    "trx_l7d",
    "trx_l30d",
    "trx_l60d",
    "trx_l90d",
    "sessions_l7d",
    "sessions_l30d",
    "sessions_l60d",
    "sessions_l90d",
    "read_noti_l7d",
    "read_noti_l30d",
    "read_noti_l60d",
    "read_noti_l90d"
]

trx_missing_rows = user_list[user_list["trx_l7d"].isna()].copy()

behavior_missing_rate = (
    trx_missing_rows[behavior_cols]
    .isna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
    .to_frame("missing_percent")
)

behavior_missing_rate

,missing_percent
trx_recency_l90d,100.000000
trx_l7d,100.000000
trx_l30d,100.000000
trx_l60d,100.000000
trx_l90d,100.000000
sessions_l7d,90.410189
sessions_l30d,90.410189
sessions_l60d,90.410189
sessions_l90d,90.410189
read_noti_l7d,90.410189


### Missing pattern-ийн ажиглалт

`trx_*` болон `avg_trx_amount_*` feature-үүдийн missing pattern
хоорондоо бүрэн нийцэж байна. `trx_*` missing байгаа бүх
observation-д corresponding `avg_trx_amount_*` мөн missing байна.

Мөн `trx_*` болон `trx_recency_l90d` feature-үүд хоорондоо ижил
missing pattern-тэй байна. Харин `sessions_*` болон `read_noti_*`
feature-үүдийн missing pattern бүрэн давхцахгүй байна.

Иймээс transaction-related missing утгуудыг нэг бүлэг болгон авч
үзэх боломжтой боловч бүх behavioral feature-ийг ижил аргаар
боловсруулахгүй.

In [27]:
duplicate_count = user_list.duplicated().sum()

credit_counts = user_list["credit_id"].value_counts()

print("Exact duplicate rows:", duplicate_count)
print("Total observations:", len(user_list))
print("Unique credit_id:", user_list["credit_id"].nunique())
print("Credits with multiple events:", (credit_counts > 1).sum())
print("Maximum events for one credit:", credit_counts.max())

Exact duplicate rows: 0
Total observations: 61200
Unique credit_id: 46516
Credits with multiple events: 12605
Maximum events for one credit: 6


In [28]:
credit_event_summary = (
    user_list
    .groupby("credit_id")
    .agg(
        event_count=("date_dpd_31", "size"),
        first_dpd31=("date_dpd_31", "min"),
        last_dpd31=("date_dpd_31", "max")
    )
    .reset_index()
)

credit_event_summary["event_span_days"] = (
    credit_event_summary["last_dpd31"]
    - credit_event_summary["first_dpd31"]
).dt.days

credit_event_summary.describe()

,credit_id,event_count,first_dpd31,last_dpd31,event_span_days
count,4.651600e+04,46516.000000,46516,46516,46516.000000
mean,2.282606e+06,1.315676,2026-03-03 16:31:40.765327872,2026-03-30 00:48:10.153925376,26.344785
min,2.000111e+06,1.000000,2025-12-16 00:00:00,2025-12-16 00:00:00,0.000000
25%,2.150116e+06,1.000000,2026-01-15 00:00:00,2026-02-15 00:00:00,0.000000
50%,2.266380e+06,1.000000,2026-02-15 00:00:00,2026-04-15 00:00:00,0.000000
75%,2.420579e+06,2.000000,2026-04-15 00:00:00,2026-05-16 00:00:00,59.000000
max,2.623202e+06,6.000000,2026-06-15 00:00:00,2026-06-15 00:00:00,181.000000
std,1.611693e+05,0.559145,NaN,NaN,47.683065


In [29]:
event_distribution = (
    credit_counts
    .value_counts()
    .sort_index()
    .rename_axis("number_of_events")
    .reset_index(name="number_of_credits")
)

event_distribution

,number_of_events,number_of_credits
0,1,33911
1,2,10685
2,3,1769
3,4,144
4,5,6
5,6,1


In [30]:
tmp = user_list.sort_values(
    ["credit_id", "date_dpd_31"]
).copy()

tmp["days_since_previous_event"] = (
    tmp.groupby("credit_id")["date_dpd_31"]
    .diff()
    .dt.days
)

tmp["days_since_previous_event"].describe()

count    14684.000000
mean        83.455053
std         33.475044
min         28.000000
25%         61.000000
50%         89.000000
75%         92.000000
max        181.000000
Name: days_since_previous_event, dtype: float64

In [31]:
credit_event_summary.describe()

,credit_id,event_count,first_dpd31,last_dpd31,event_span_days
count,4.651600e+04,46516.000000,46516,46516,46516.000000
mean,2.282606e+06,1.315676,2026-03-03 16:31:40.765327872,2026-03-30 00:48:10.153925376,26.344785
min,2.000111e+06,1.000000,2025-12-16 00:00:00,2025-12-16 00:00:00,0.000000
25%,2.150116e+06,1.000000,2026-01-15 00:00:00,2026-02-15 00:00:00,0.000000
50%,2.266380e+06,1.000000,2026-02-15 00:00:00,2026-04-15 00:00:00,0.000000
75%,2.420579e+06,2.000000,2026-04-15 00:00:00,2026-05-16 00:00:00,59.000000
max,2.623202e+06,6.000000,2026-06-15 00:00:00,2026-06-15 00:00:00,181.000000
std,1.611693e+05,0.559145,NaN,NaN,47.683065


In [32]:
event_distribution

,number_of_events,number_of_credits
0,1,33911
1,2,10685
2,3,1769
3,4,144
4,5,6
5,6,1


In [33]:
tmp["days_since_previous_event"].describe()

count    14684.000000
mean        83.455053
std         33.475044
min         28.000000
25%         61.000000
50%         89.000000
75%         92.000000
max        181.000000
Name: days_since_previous_event, dtype: float64

# 3. Temporal & Data Leakage Audit

Model нь `date_dpd_31` буюу DPD31 болсон мөчид available байсан
information дээр үндэслэн `is_dpd_91`-ийг таамаглах ёстой.

Иймээс feature бүрийг prediction point-той уялдуулан шалгаж,
future information болон identifier-үүдийг model-д оруулахгүй.

`data_description`-ийн дагуу `user_list`-ийн бусад feature-үүд
`date_dpd_31`-ийг Day 0 буюу cutoff point гэж үзэн тодорхойлогдсон.

In [34]:
description_df = pd.read_excel(
    FILE_PATH,
    sheet_name="data_description",
    engine="openpyxl"
)

description_df.head()

,Sheet Name,Columns,Column Description
0,payment,created_month,Month
1,NaN,credit_id,Unique identifier for the loan account/credit ...
2,NaN,total_paid_amt,Total paid amount MNT
3,overdue,created_month,Month
4,NaN,credit_id,Unique identifier for the loan account/credit ...


In [35]:
description_df.columns

Index(['Sheet Name', 'Columns', 'Column Description'], dtype='object')

In [36]:
description_df["sheet_group"] = description_df["Sheet Name"].ffill()

description_df[["sheet_group", "Columns", "Column Description"]]

,sheet_group,Columns,Column Description
0,payment,created_month,Month
1,payment,credit_id,Unique identifier for the loan account/credit ...
2,payment,total_paid_amt,Total paid amount MNT
3,overdue,created_month,Month
4,overdue,credit_id,Unique identifier for the loan account/credit ...
...,...,...,...
92,user_list,number_change_cnt_l30d,Count of SIM card or device hardware identifie...
93,user_list,number_change_cnt_l60d,Count of SIM card or device hardware identifie...
94,user_list,number_change_cnt_l90d,Count of SIM card or device hardware identifie...
95,user_list,number_change_cnt,Lifetime count of SIM card or device hardware ...


In [37]:
description_df.groupby("sheet_group").size()

sheet_group
balance       9
overdue      42
payment       3
user_list    43
dtype: int64

In [38]:
description_df["sheet_group"].value_counts()

sheet_group
user_list    43
overdue      42
balance       9
payment       3
Name: count, dtype: int64

In [39]:
user_features = description_df[
    description_df["sheet_group"] == "user_list"
].copy()

user_features[
    ["Columns", "Column Description"]
]

,Columns,Column Description
54,created_month,Month
55,date_dpd_31,Exact date when the borrower's delinquency rea...
56,credit_id,Unique identifier for the loan account/credit ...
57,limit_dpd31,Credit line limit at the moment DPD 31 was tri...
58,balance_dpd31,Outstanding balance utilized at the moment DPD...
59,limit_ondue,Credit line limit on the payment due date
60,balance_ondue,Outstanding balance utilized on the payment du...
61,invoiced_unpaid_amt,Total billed/requested invoice amount that rem...
62,total_payment_amt,Total payment amount successfully received dur...
63,ostype,"Smartphone operating system (e.g., iOS, Android)"


In [40]:
feature_audit = user_features[
    ["Columns", "Column Description"]
].copy()

feature_audit["role"] = "Candidate Feature"
feature_audit["leakage_status"] = "To Review"

feature_audit.loc[
    feature_audit["Columns"] == "credit_id",
    ["role", "leakage_status"]
] = ["Exclude", "Identifier"]

feature_audit.loc[
    feature_audit["Columns"] == "date_dpd_31",
    ["role", "leakage_status"]
] = ["Exclude", "Prediction Point"]

feature_audit.loc[
    feature_audit["Columns"] == "is_dpd_91",
    ["role", "leakage_status"]
] = ["Target", "Target Variable"]

feature_audit

,Columns,Column Description,role,leakage_status
54,created_month,Month,Candidate Feature,To Review
55,date_dpd_31,Exact date when the borrower's delinquency rea...,Exclude,Prediction Point
56,credit_id,Unique identifier for the loan account/credit ...,Exclude,Identifier
57,limit_dpd31,Credit line limit at the moment DPD 31 was tri...,Candidate Feature,To Review
58,balance_dpd31,Outstanding balance utilized at the moment DPD...,Candidate Feature,To Review
59,limit_ondue,Credit line limit on the payment due date,Candidate Feature,To Review
60,balance_ondue,Outstanding balance utilized on the payment du...,Candidate Feature,To Review
61,invoiced_unpaid_amt,Total billed/requested invoice amount that rem...,Candidate Feature,To Review
62,total_payment_amt,Total payment amount successfully received dur...,Candidate Feature,To Review
63,ostype,"Smartphone operating system (e.g., iOS, Android)",Candidate Feature,To Review


In [41]:
feature_audit = user_features[
    ["Columns", "Column Description"]
].copy()

feature_audit["role"] = "Candidate Feature"
feature_audit["leakage_status"] = "Potentially safe"

# Excluded columns
feature_audit.loc[
    feature_audit["Columns"] == "credit_id",
    ["role", "leakage_status"]
] = ["Exclude", "Identifier"]

feature_audit.loc[
    feature_audit["Columns"] == "date_dpd_31",
    ["role", "leakage_status"]
] = ["Exclude", "Prediction Point"]

feature_audit.loc[
    feature_audit["Columns"] == "is_dpd_91",
    ["role", "leakage_status"]
] = ["Target", "Target Variable"]

# Features requiring additional review
feature_audit.loc[
    feature_audit["Columns"].isin(
        ["created_month", "total_payment_amt"]
    ),
    "leakage_status"
] = "Needs temporal review"

feature_audit

,Columns,Column Description,role,leakage_status
54,created_month,Month,Candidate Feature,Needs temporal review
55,date_dpd_31,Exact date when the borrower's delinquency rea...,Exclude,Prediction Point
56,credit_id,Unique identifier for the loan account/credit ...,Exclude,Identifier
57,limit_dpd31,Credit line limit at the moment DPD 31 was tri...,Candidate Feature,Potentially safe
58,balance_dpd31,Outstanding balance utilized at the moment DPD...,Candidate Feature,Potentially safe
59,limit_ondue,Credit line limit on the payment due date,Candidate Feature,Potentially safe
60,balance_ondue,Outstanding balance utilized on the payment du...,Candidate Feature,Potentially safe
61,invoiced_unpaid_amt,Total billed/requested invoice amount that rem...,Candidate Feature,Potentially safe
62,total_payment_amt,Total payment amount successfully received dur...,Candidate Feature,Needs temporal review
63,ostype,"Smartphone operating system (e.g., iOS, Android)",Candidate Feature,Potentially safe


In [42]:
feature_audit.groupby(
    ["role", "leakage_status"]
).size()

role               leakage_status       
Candidate Feature  Needs temporal review     2
                   Potentially safe         38
Exclude            Identifier                1
                   Prediction Point          1
Target             Target Variable           1
dtype: int64

### Feature availability-ийн эхний үнэлгээ

`user_list`-ийн feature-үүдийг `date_dpd_31`-ийг prediction point
болгон авч үзэн шалгав.

`credit_id` нь identifier, `date_dpd_31` нь prediction point,
`is_dpd_91` нь target тул model-ийн input-д оруулахгүй.

Historical behavioral болон user-level feature-үүдийг candidate feature
гэж үзэв. Харин `created_month` болон `total_payment_amt` нь хугацааны
хамаарлыг нэмэлтээр шалгах шаардлагатай гэж үзэв.

`payment`, `overdue`, `balance` sheet-үүдийг baseline model-д шууд
merge хийхгүй бөгөөд ашиглах тохиолдолд prediction point-оос өмнөх
мэдээллээр feature үүсгэнэ.

In [43]:
payment_df = pd.read_excel(
    FILE_PATH,
    sheet_name="payment",
    engine="openpyxl"
)

payment_df["created_month"].min(), payment_df["created_month"].max()

(np.int64(202407), np.int64(202606))